# SITCOM-1399: Adding Earthquake events without needs of VMS data, to the TN81, on vibration on M1M3

2025-01-08 HyeYun Park

**Descripción**

Sub-ticket related to the [SITCOM-918: Write technote on hardpoint oscillations during tma slews](https://rubinobs.atlassian.net/browse/SITCOM-918), to study earthquake events seen by M1M3. 
We can read the TN81 [here](https://sitcomtn-081.lsst.io/v/SITCOM-918/index.html).

This notebook is to update SITCOM-1399 to avoid downloading big data of VMS, and to update new event below which was big and near from the Rubin observatory which faulted the M1M3 during the event.
- 2024-11-26 UTC 12:33:09

## Imports

In [ ]:
import sys, time, os, asyncio
import scipy.stats as stats
from scipy.signal import find_peaks
from scipy import signal
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.time import Time
from lsst.summit.utils.tmaUtils import TMAEventMaker, TMAState
from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient, clipDataToEvent, calcNextDay
import matplotlib.dates as mdates
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
# Functions to get data
# Modified functions of ticket SITCOM-761

key_m1m3_dict={'1 X': 'm1m3_x_1', 
            '1 Y': 'm1m3_y_1', 
            '1 Z': 'm1m3_z_1', 
            '2 X': 'm1m3_x_2', 
            '2 Y': 'm1m3_z_2', # note these two have been 
            '2 Z': 'm1m3_y_2', # switched pending SUMMIT-7911
            '3 X': 'm1m3_x_3', 
            '3 Y': 'm1m3_y_3', 
            '3 Z': 'm1m3_z_3'
            }

def get_efd_data(begin, end, client):

    """Extract all the MTMount data from the EFD and add to dict.

    Args:
        begin (str): The start time of the query.
        end (str): The end time of the query.
        client (object): influx client

    Returns:
        dict: A dictionary containing the MTMount data.
    """

    query_dict = {}

    query_dict["el"] = getEfdData(
        client,
        "lsst.sal.MTMount.elevation",
        columns=["private_sndStamp", "private_efdStamp", "actualPosition", "actualVelocity", "actualTorque"],
        begin=begin,
        end=end,
        prePadding=0,
        postPadding=0,
        warn=False,
    )
    query_dict["az"] = getEfdData(
        client,
        "lsst.sal.MTMount.azimuth",
        columns=["private_sndStamp", "private_efdStamp", "actualPosition", "actualVelocity", "actualTorque"],
        begin=begin,
        end=end,
        prePadding=0,
        postPadding=0,
        warn=False,
    )
    return query_dict


In [ ]:
def get_freq_psd(vals, timestep):

    """
    Calculates the frequency power spectrum of a signal.

    Args:
        vals (np.array): The signal values.
        timestep (float): The time step between samples.

    Returns:
        tuple: The frequencies and power spectral density.
    """

    # Remove the mean from the signal.
    meanval = np.mean(vals)
    signal = vals - meanval

    # Calculate the length of the signal.
    N = len(signal)

    # Calculate the power spectral density.
    psd = np.abs(np.fft.rfft(np.array(signal) * 1)) ** 2

    # Calculate the frequencies.
    frequencies = np.fft.rfftfreq(N, timestep)

    return (frequencies, psd)

def get_peak_points(freq, psd, height=0.01):
    """
    Get the peak points of the power spectral density (PSD).

    Args:
        freq (numpy.ndarray): The frequency vector.
        psd (numpy.ndarray): The power spectral density.
        height (float): The minimum peak height.

    Returns:
        numpy.ndarray: The peak points.
    """

    # Find the peak indices and heights.
    peak_ind, peak_dict = find_peaks(psd, height=height)
    peaks = freq[peak_ind]

    # If there are no peaks, return None.
    if len(peaks) < 1:
        return None

    # Find the sub-peaks within each group of peaks that are close in frequency.
    points = []
    for i, peak in enumerate(peaks):
        sel = (abs(peaks - peak) < 1)
        sub_peaks = peaks[sel]
        sub_heights = peak_dict['peak_heights'][sel]
        points.append(sub_peaks[np.argmax(sub_heights)])

    # Return the unique peak points.
    return np.unique(np.array(points))

## Analysis for the evening's event:  2024-11-26 UTC 12:33:09

In [ ]:
client=makeEfdClient()
begin_time=Time('2024-11-26 12:30:00', format="iso", scale="utc")
end_time=Time('2024-11-26 12:40:00', format="iso", scale="utc")
efd_dict=get_efd_data(begin_time, end_time, client)

### HP Forces

The limit required to keep the mirror safe during a mag 6 earthquake must be less than 3000 N. Let's study the and HP forces along the same axes and compare them with the accelerations.

In [ ]:
# Select data from a given date
dayObs = 20241126
eventMaker = TMAEventMaker()
event = eventMaker.getEvents(dayObs)

#### Earthquake events using the EFD client

In [ ]:
from lsst_efd_client import EfdClient
client = EfdClient('usdf_efd', db_name="lsst.backpack")

topics = await client.get_topics()
topics

In [ ]:
query = f'''SELECT * FROM "lsst.backpack.usgs_earthquake_data"'''
await client.influx_client.query(query)

In [ ]:
data = getEfdData(client, "lsst.backpack.usgs_earthquake_data", dayObs=20241126)
data

In [ ]:
# Pick which earthquake to have a look. This case first ([0]) one  on the list is picked.
data.index[0]

In [ ]:
client = EfdClient("usdf_efd")

In [ ]:
df_hp = getEfdData(
    client, "lsst.sal.MTM1M3.hardpointActuatorData", begin=begin_time, end=end_time
)

In [ ]:
def compare_mount_hardpoints(
    df_mtmount_ele,
    df_mtmount_azi,
    df_hp,
    begin,
    end,
):
    fig, axs = plt.subplots(1, 1, dpi=125, figsize=(15, 8))
    ax = axs  # [0]
    df_plot = df_hp["measuredForce0"][begin:end]
    ax.plot(df_plot, color="red", lw="0.5", label="HP 0")
    df_plot = df_hp["measuredForce1"][begin:end]
    ax.plot(df_plot, color="blue", lw="0.5", label="HP 1")
    df_plot = df_hp["measuredForce2"][begin:end]
    ax.plot(df_plot, color="black", lw="0.5", label="HP 2")
    df_plot = df_hp["measuredForce3"][begin:end]
    ax.plot(df_plot, color="green", lw="0.5", label="HP 3")
    df_plot = df_hp["measuredForce4"][begin:end]
    ax.plot(df_plot, color="orange", lw="0.5", label="HP 4")
    df_plot = df_hp["measuredForce5"][begin:end]
    ax.plot(df_plot, color="yellow", lw="0.5", label="HP 5")
    ax.set_ylabel("HP Force \n[N]")
    # ax.legend()
    ax2 = ax.twinx()
    # ax = axs[1]
    df_plot = df_mtmount_ele["actualPosition"][begin:end]
    ax2.plot(df_plot, color="green", lw="0.5", label="el")
    # ax.axvline(begin, lw="0.5", c="k", label="Slew start")
    # ax.axvline(end, lw="0.5", c="b", label="Slew stop")

    # ax = axs[2]
    df_plot = df_mtmount_azi["actualPosition"][begin:end]
    ax2.plot(df_plot, color="red", lw="0.5", label="az")
    ax2.set_ylabel("TMA El & Azimuth \nPosition\n[deg]")

    ax2.set_xlabel("UTC")
    ax2.legend()
    fig.autofmt_xdate()
    fig.subplots_adjust(hspace=1)
    fig.suptitle(t0)
    fig.tight_layout()

In [ ]:
df_hp

In [ ]:
df_hp.index

In [ ]:
def compare_mount_hardpoints(
    df_mtmount_ele,
    df_mtmount_azi,
    df_hp,
    begin,
    end
):
    fig, axs = plt.subplots(1, 1, dpi=125, figsize=(15, 8))
    ax = axs  # [0]
    df_plot = df_hp["measuredForce0"][begin:end]
    ax.plot(df_plot, color="red", lw="0.5", label="HP 0")
    df_plot = df_hp["measuredForce1"][begin:end]
    ax.plot(df_plot, color="blue", lw="0.5", label="HP 1")
    df_plot = df_hp["measuredForce2"][begin:end]
    ax.plot(df_plot, color="black", lw="0.5", label="HP 2")
    df_plot = df_hp["measuredForce3"][begin:end]
    ax.plot(df_plot, color="green", lw="0.5", label="HP 3")
    df_plot = df_hp["measuredForce4"][begin:end]
    ax.plot(df_plot, color="orange", lw="0.5", label="HP 4")
    df_plot = df_hp["measuredForce5"][begin:end]
    ax.plot(df_plot, color="yellow", lw="0.5", label="HP 5")
    ax.set_ylabel("HP Force \n[N]")
    # ax.legend()
    ax2 = ax.twinx()
    # ax = axs[1]
    df_plot = df_mtmount_ele["actualPosition"][begin:end]
    ax2.plot(df_plot, color="green", lw="0.5", label="el")
    # ax.axvline(begin, lw="0.5", c="k", label="Slew start")
    # ax.axvline(end, lw="0.5", c="b", label="Slew stop")

    # ax = axs[2]
    df_plot = df_mtmount_azi["actualPosition"][begin:end]
    ax2.plot(df_plot, color="red", lw="0.5", label="az")
    ax2.set_ylabel("TMA El & Azimuth \nPosition\n[deg]")

    ax2.set_xlabel("UTC")
    ax2.legend()
    fig.autofmt_xdate()
    fig.subplots_adjust(hspace=1)
    fig.suptitle(f"Earthquake detected at {data.index[0]}")
    fig.tight_layout()

In [ ]:
df_mtmount_ele = getEfdData(
    client,
    "lsst.sal.MTMount.elevation", begin=begin_time, end=end_time
)
df_mtmount_azi = getEfdData(
    client,
    "lsst.sal.MTMount.azimuth", begin=begin_time, end=end_time
)
df_hp = getEfdData(
    client, "lsst.sal.MTM1M3.hardpointActuatorData", begin=begin_time, end=end_time)

In [ ]:
start_slew=data.index[0]
t0 = pd.to_datetime(start_slew.value, utc=True)

In [ ]:
import datetime
t1=(t0-datetime.timedelta(seconds=100)).strftime('%Y-%m-%dT%H:%M:%SZ')
t2=(t0+datetime.timedelta(seconds=260)).strftime('%Y-%m-%dT%H:%M:%SZ')

In [ ]:
%matplotlib inline
compare_mount_hardpoints(
    df_mtmount_ele,
    df_mtmount_azi,
    df_hp,
    t1,
    t2
)

In [ ]:
def compare_mount_hardpoints_separate_plots(
    df_mtmount_ele,
    df_mtmount_azi,
    df_hp,
    begin,
    end,
):
    fig, axs = plt.subplots(3, 1, dpi=125, figsize=(15, 8))
    ax = axs[0]
    df_plot = df_hp["measuredForce0"][begin:end]
    ax.plot(df_plot, color="red", lw="0.5", label="HP 0")
    df_plot = df_hp["measuredForce1"][begin:end]
    ax.plot(df_plot, color="blue", lw="0.5", label="HP 1")
    df_plot = df_hp["measuredForce2"][begin:end]
    ax.plot(df_plot, color="black", lw="0.5", label="HP 2")
    df_plot = df_hp["measuredForce3"][begin:end]
    ax.plot(df_plot, color="green", lw="0.5", label="HP 3")
    df_plot = df_hp["measuredForce4"][begin:end]
    ax.plot(df_plot, color="orange", lw="0.5", label="HP 4")
    df_plot = df_hp["measuredForce5"][begin:end]
    ax.plot(df_plot, color="yellow", lw="0.5", label="HP 5")
    ax.set_ylabel("HP Force \n[N]")

    ax = axs[1]
    df_plot = df_mtmount_ele["actualPosition"][begin:end]
    ax.plot(df_plot, color="green", lw="0.5")
    ax.set_ylabel("TMA Elevation \nPosition\n[deg]")
    # ax.axvline(begin, lw="0.5", c="k", label="Slew start")
    # ax.axvline(end, lw="0.5", c="b", label="Slew stop")

    ax = axs[2]
    df_plot = df_mtmount_azi["actualPosition"][begin:end]
    ax.plot(df_plot, color="red", lw="0.5")
    ax.set_ylabel("TMA Azimuth \nPosition\n[deg]")

    ax.set_xlabel("UTC")
    fig.autofmt_xdate()
    fig.subplots_adjust(hspace=1)
    fig.suptitle(t0)
    fig.legend()
    fig.tight_layout()

In [ ]:
%matplotlib inline
compare_mount_hardpoints_separate_plots(
    df_mtmount_ele,
    df_mtmount_azi,
    df_hp,
    t1,
    t2,
)